Right now, the data contains individual timestamps for every step of an order's journey. We need to compress these transactional logs into a single, comprehensive row per customer.

In [1]:
import os
import pandas as pd
import numpy as np

RAW_DATA_DIR = "../data/raw/"

def engineer_customer_features(base_path):
    print("Initializing Phase 2 Feature Engineering Pipeline...")
    
    # 1. Reload core tables from Phase 1
    customers = pd.read_csv(os.path.join(base_path, "olist_customers_dataset.csv"), usecols=["customer_id", "customer_unique_id"])
    orders = pd.read_csv(os.path.join(base_path, "olist_orders_dataset.csv"))
    payments = pd.read_csv(os.path.join(base_path, "olist_order_payments_dataset.csv")).groupby("order_id")["payment_value"].sum().reset_index()
    
    # 2. Load Phase 2 contextual tables (Reviews and Items for delivery logistics)
    reviews = pd.read_csv(os.path.join(base_path, "olist_order_reviews_dataset.csv"), usecols=["order_id", "review_score"])
    items = pd.read_csv(os.path.join(base_path, "olist_order_items_dataset.csv"), usecols=["order_id", "freight_value"])
    
    # Aggregate items to get total freight paid per order
    freight_aggregated = items.groupby("order_id")["freight_value"].sum().reset_index()
    # Take the average review score if an order received multiple reviews
    reviews_aggregated = reviews.groupby("order_id")["review_score"].mean().reset_index()

    # 3. Compile base relational frame
    df = pd.merge(orders, customers, on="customer_id", how="inner")
    df = pd.merge(df, payments, on="order_id", how="left")
    df = pd.merge(df, freight_aggregated, on="order_id", how="left")
    df = pd.merge(df, reviews_aggregated, on="order_id", how="left")
    
    # 4. Filter out systematically invalid data
    valid_statuses = ['delivered', 'shipped', 'invoiced', 'processing']
    df = df[df['order_status'].isin(valid_statuses)].copy()
    
    # 5. Typecast Timestamps safely
    timestamp_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
    for col in timestamp_cols:
        df[col] = pd.to_datetime(df[col])
        
    # Drop rows missing critical transaction dates
    df = df.dropna(subset=['order_purchase_timestamp'])
    
    print("Constructing engineering operational metrics...")
    # Feature A: Delivery Performance (Actual vs Estimated delay in days)
    # Intuition: Positive values mean an order arrived late; negative means early.
    df['delivery_delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
    df['delivery_delay_days'] = df['delivery_delay_days'].fillna(0) # Assume on-time if missing for active shipments
    
    # 6. Establish the Snapshot Date for Recency calculation
    # Intuition: Olist is a historical dataset. Using today's date would make every customer look 
    # hundreds of days inactive. We anchor to max transaction date + 1 day.
    snapshot_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
    
    print(f"Dataset snapshot date anchor: {snapshot_date.strftime('%Y-%m-%d')}")
    
    print("Structuring aggregates to Unique Customer level...")
    # 7. Ultimate RFM + Operations Rollup
    customer_master = df.groupby("customer_unique_id").agg({
        'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days, # Recency
        'order_id': 'count',                                                 # Frequency
        'payment_value': 'sum',                                              # Monetary Value
        'freight_value': 'sum',                                              # Operational overhead
        'review_score': 'mean',                                              # Satisfaction Anchor
        'delivery_delay_days': 'mean'                                        # Logistics Experience
    }).reset_index()
    
    # Clean column structures
    customer_master.rename(columns={
        'order_purchase_timestamp': 'recency',
        'order_id': 'frequency',
        'payment_value': 'monetary',
        'freight_value': 'total_freight',
        'review_score': 'avg_review_score'
    }, inplace=True)
    
    # Fill remaining missing fields safely
    customer_master['avg_review_score'] = customer_master['avg_review_score'].fillna(customer_master['avg_review_score'].median())
    customer_master['monetary'] = customer_master['monetary'].fillna(0)
    customer_master['total_freight'] = customer_master['total_freight'].fillna(0)
    
    print(f"Engineering complete. Master Customer Profile Shape: {customer_master.shape}")
    return customer_master

# Execute
df_customer_features = engineer_customer_features(RAW_DATA_DIR)
print(df_customer_features.head())

Initializing Phase 2 Feature Engineering Pipeline...
Constructing engineering operational metrics...
Dataset snapshot date anchor: 2018-09-04
Structuring aggregates to Unique Customer level...
Engineering complete. Master Customer Profile Shape: (94984, 7)
                 customer_unique_id  recency  frequency  monetary  \
0  0000366f3b9a7992bf8c76cfdf3221e2      116          1    141.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f      119          1     27.19   
2  0000f46a3911fa3c0805444483337064      542          1     86.22   
3  0000f6ccb0745a6a4b88665a16c9f078      326          1     43.62   
4  0004aac84e0df4da2b147fca70cf8255      293          1    196.89   

   total_freight  avg_review_score  delivery_delay_days  
0          12.00               5.0                 -5.0  
1           8.29               4.0                 -5.0  
2          17.22               3.0                 -2.0  
3          17.63               4.0                -12.0  
4          16.89               5.0    

In [2]:
print(df_customer_features[['recency', 'frequency', 'monetary']].describe())

            recency     frequency      monetary
count  94984.000000  94984.000000  94984.000000
mean     243.441295      1.033858    165.693252
std      152.998564      0.210810    226.746987
min        1.000000      1.000000      0.000000
25%      119.000000      1.000000     63.100000
50%      224.000000      1.000000    107.900000
75%      352.000000      1.000000    182.942500
max      729.000000     16.000000  13664.080000


In [3]:
# Create the processed directory if it doesn't exist
os.makedirs("../data/processed/", exist_ok=True)

# Save to a highly optimized Parquet file
df_customer_features.to_parquet("../data/processed/phase2_customer_features.parquet", index=False)
print("Phase 2 data safely serialized to disk!")

Phase 2 data safely serialized to disk!
